# VLM 하자 설명 배치화 검증 — 이미지 1장, 번호별 일괄 설명

### 이 노트북이 답하려는 질문
지금까지 검증(§CH009/CH011)된 건 **"하자 1건 크롭 -> VLM 호출 1번"** 뿐이다. 프로덕션(`worker/models/quality_vlm/describer.py`)이 이 방식이라 finding이 6~16건이면 그만큼 호출한다(건당 3~9초). 이 노트북은 그 반대 극단 — **"이미지 1장(번호별 하자 여러 건) -> VLM 호출 1번"**을 테스트한다.

1. Qwen3-VL이 넘버링 bbox 이미지에서 **번호를 안 헷갈리고** 각각 구분해서 답하는가.
2. 배치 방식의 설명 품질이 건별(크롭) 방식과 비교해 떨어지지 않는가.
3. 실제로 호출 시간이 줄어드는가(배치 1회 vs 건별 N회).

### 실행 순서
0. 의존성 설치 -> HF 로그인 -> GPU 확인 -> 설정 -> SAM3/VLM 모델 로드 -> 테스트 이미지
1. Step 1 — 실제 하자 탐지 + 넘버링 bbox 이미지 생성(이미 검증된 SAM3 파이프라인 재사용)
2. Step 2 — 배치 프롬프트 1회 호출 + 파싱
3. Step 3 — 건별(크롭) 방식과 나란히 비교
4. Step 4 — 호출 시간 비교(배치 1회 vs 건별 N회)
5. 이미지 전체 배치 실행 + 결과 표

**프로덕션(`worker/models/quality_vlm/`)은 건드리지 않는다.** 여기서 배치 방식이 안전하다고 확인되면 별도 계획 문서로 `describer.py` 반영 여부를 결정한다.


## 0. 의존성 설치


In [ ]:
# SAM3(탐지)와 Qwen3-VL(설명) 두 세트 다 필요하다 — CH011에서 같은 conda 환경(qual)에
# 동시 설치 가능함을 이미 확인했다.
%pip install -q -r "../sam3-condition-check/requirements-sam3.txt"
%pip install -q -r "requirements-vlm.txt"


## 0-1. Hugging Face 로그인 (SAM3 gated repo)


In [ ]:
import os

from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("HF_TOKEN 환경변수로 로그인했습니다.")
else:
    print("HF_TOKEN이 없습니다 — huggingface-cli login을 미리 실행했다면 그대로 진행됩니다.")


## 0-2. CUDA / GPU 확인


In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


## 0-3. 경로·설정


In [ ]:
import sys

# 공유 유틸(qc_shared.py)은 이 폴더의 부모(validation/QUAL/)에 있다.
sys.path.insert(0, "..")
# SAM3 호출/크롭 유틸과 검증된 bbox 파이프라인(garment 탐지 -> 하자 탐지 -> 후처리)은
# 새로 만들지 않고 형제 스파이크 폴더의 구현을 그대로 재사용한다.
sys.path.insert(0, "../sam3-parts-segment-check")
sys.path.insert(0, "../sam3-border-overlay-check")

import time

import matplotlib.pyplot as plt
from PIL import Image

import qc_shared
from border_overlay_check import run_one
from sam3_caller import Sam3Caller

import batch_prompts
import model_registry
from loaders import build_defect_prompt, describe_defect, generate_from_image, load_qwen_vl

qc_shared.use_korean_font()  # 그래프 제목 한글이 두부(□□)로 깨지지 않게


## 0-4. SAM3 모델 로드 (하자 탐지용)


In [ ]:
sam3_caller = Sam3Caller()
sam3_caller._ensure_loaded()


## 0-5. VLM 모델 로드 (설명 생성용)


In [ ]:
print(f"모델: {model_registry.ACTIVE_VLM} ({model_registry.VLM_MODELS[model_registry.ACTIVE_VLM]})")
vlm_bundle = load_qwen_vl(
    model_registry.VLM_MODELS[model_registry.ACTIVE_VLM],
    quantization=model_registry.ACTIVE_QUANTIZATION,
)
print("VLM 로드 완료")


## 0-6. 테스트 이미지 로드


In [ ]:
image_paths = qc_shared.list_images()
assert image_paths, "sample_images/ 에 이미지가 없습니다 (qc_shared.SHARED_IMAGE_DIR 참고)."
print(f"이미지 {len(image_paths)}장: {[p.name for p in image_paths]}")

# §1~§4는 이 기준 이미지 1장으로 단계별로 확인한다. §5에서 전체 배치를 돈다.
# 하자가 여러 건 잡히는 이미지를 고르는 게 배치 검증에 유리하다 — 필요하면 인덱스를 바꾼다.
demo_path = image_paths[0]
demo_image = Image.open(demo_path).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(demo_image)
plt.title(f"기준 이미지: {demo_path.name}")
plt.axis("off")
plt.show()


## 1. Step 1 — 실제 하자 탐지 + 넘버링 bbox 이미지 생성

`sam3-border-overlay-check`의 검증된 파이프라인(옷 1회 세그먼트 -> 하자 탐지 -> 공통 후처리)을 그대로 재사용한다 — 이 노트북에서 새로 탐지 로직을 만들지 않는다. 넘버링 bbox 이미지는 프로덕션(CH007)과 같은 방식(`qc_shared.draw_numbered_boxes`, 번호만 표시)으로 만든다.


In [ ]:
result = run_one(sam3_caller, demo_image)
kept = result["kept"]
print(f"하자 {len(kept)}건 탐지, garment_found={result['garment_found']}")

numbered_image_arr = qc_shared.draw_numbered_boxes(demo_image, kept)
numbered_image = Image.fromarray(numbered_image_arr)

labels_by_no = {inst["no"]: qc_shared.KOREAN_LABELS.get(inst["type"], inst["type"]) for inst in kept}
print("번호별 하자 유형:", labels_by_no)

plt.figure(figsize=(5, 6))
plt.imshow(numbered_image)
plt.title("넘버링 bbox 이미지 (VLM 배치 입력)")
plt.axis("off")
plt.show()


## 2. Step 2 — 배치 프롬프트 1회 호출 + 파싱

번호별 하자 유형만 알려주고(위치는 여전히 묻지 않는다), 번호 순서대로 한 문장씩 설명을 요청한다. 형식을 벗어나거나 번호를 빠뜨리면 `parse_batch_response`가 그 번호를 `None`으로 남긴다 — 조용히 넘어가지 않는다.


In [ ]:
batch_prompt_text = batch_prompts.build_batch_defect_prompt(labels_by_no)
print(batch_prompt_text)
print("---")

batch_started = time.perf_counter()
batch_raw_response = generate_from_image(
    vlm_bundle, numbered_image, batch_prompt_text, max_new_tokens=80 * max(1, len(kept))
)
batch_elapsed = time.perf_counter() - batch_started

print(f"원문 응답 ({batch_elapsed:.1f}초):\n{batch_raw_response}")

batch_parsed = batch_prompts.parse_batch_response(batch_raw_response, list(labels_by_no))
missing = batch_prompts.missing_numbers(batch_parsed)
print(f"\n파싱 결과: {batch_parsed}")
if missing:
    print(f"[주의] 번호 {missing}는 배치 응답에서 못 찾았습니다.")


## 3. Step 3 — 건별(크롭) 방식과 나란히 비교

같은 findings를 기존 검증된 방식(하자 1건 크롭 -> 개별 호출, `loaders.describe_defect`)으로도 돌려서 번호별로 나란히 비교한다. 크롭은 프로덕션과 같은 여백·최소크기 공식(`batch_prompts.crop_for_individual_baseline`)을 쓴다.


In [ ]:
individual_started = time.perf_counter()
individual_by_no: dict[int, str] = {}
for inst in kept:
    crop = batch_prompts.crop_for_individual_baseline(demo_image, inst["bbox"], demo_image.size)
    label = qc_shared.KOREAN_LABELS.get(inst["type"], inst["type"])
    individual_by_no[inst["no"]] = describe_defect(vlm_bundle, crop, label)
individual_elapsed = time.perf_counter() - individual_started

for no in sorted(labels_by_no):
    print(f"[{no}] {labels_by_no[no]}")
    print(f"  배치 : {batch_parsed.get(no)}")
    print(f"  건별 : {individual_by_no.get(no)}")
    print()


## 4. Step 4 — 호출 시간 비교

배치(1회) vs 건별(N회) 총 소요 시간. 배치가 실제로 이득인지는 이 숫자로 판단한다 — 파싱 정확도가 같아도 시간이 안 줄면 배치화의 의미가 없다.


In [ ]:
print(f"배치(1회 호출): {batch_elapsed:.1f}초")
print(f"건별({len(kept)}회 호출): {individual_elapsed:.1f}초")
if individual_elapsed > 0:
    print(f"배치가 건별 대비 {individual_elapsed / max(batch_elapsed, 1e-6):.1f}배 빠름")


## 5. 이미지 전체 배치 실행 + 결과 표

§1~§4를 이미지마다 반복해서 누락 번호 개수·소요 시간만 표로 모은다. 개별 이미지를 다시 자세히 보고 싶으면 `demo_path`/`demo_image`를 바꿔서 §1부터 재실행한다.


In [ ]:
import pandas as pd

rows = []
for path in image_paths:
    image = Image.open(path).convert("RGB")
    row_result = run_one(sam3_caller, image)
    row_kept = row_result["kept"]
    if not row_kept:
        rows.append(
            {"image": path.name, "finding_count": 0, "missing": 0, "batch_sec": 0.0, "individual_sec": 0.0}
        )
        continue

    row_labels = {inst["no"]: qc_shared.KOREAN_LABELS.get(inst["type"], inst["type"]) for inst in row_kept}
    row_numbered = Image.fromarray(qc_shared.draw_numbered_boxes(image, row_kept))

    t0 = time.perf_counter()
    row_response = generate_from_image(
        vlm_bundle,
        row_numbered,
        batch_prompts.build_batch_defect_prompt(row_labels),
        max_new_tokens=80 * len(row_kept),
    )
    row_batch_sec = time.perf_counter() - t0
    row_parsed = batch_prompts.parse_batch_response(row_response, list(row_labels))
    row_missing = batch_prompts.missing_numbers(row_parsed)

    t1 = time.perf_counter()
    for inst in row_kept:
        crop = batch_prompts.crop_for_individual_baseline(image, inst["bbox"], image.size)
        describe_defect(vlm_bundle, crop, row_labels[inst["no"]])
    row_individual_sec = time.perf_counter() - t1

    rows.append(
        {
            "image": path.name,
            "finding_count": len(row_kept),
            "missing": len(row_missing),
            "batch_sec": round(row_batch_sec, 1),
            "individual_sec": round(row_individual_sec, 1),
        }
    )

summary_df = pd.DataFrame(rows)
summary_df


## 결론 (실행 후 채울 것)

- 배치 응답에서 번호가 누락되거나 헷갈린 사례가 있었는가(§2/§5 `missing`):
- 배치 vs 건별 설명 품질 차이가 체감되는가(§3):
- 배치가 건별보다 실제로 빠른가, 얼마나(§4/§5):
- 채택할 만한가:

### 후속 작업
- 채택한다면 `docs/plans/AI/`에 `describer.py`/`prompts.py` 반영 설계 계획을 먼저 쓴다(CH006/CH011 선례). 여기 코드를 그대로 승격하지 않는다 — 프로덕션은 계획 문서 승인 후에만 손댄다.
